# 01 - Deploy Infrastructure

Deploy `infra/main.bicep` and write outputs to `env/.env` for later notebooks.

**Estimated time:** 45-70 minutes

> **Important:** VPN gateway provisioning alone is typically **30-45 minutes**.


## Prerequisite check - Azure VPN Client tenant consent

The VPN gateway takes 30-45 min to provision but is useless if your tenant has not consented to the Azure VPN Client multi-tenant app (`c632b3df-fb67-4d84-bdcf-b95ad541b5c8`). This cell fails fast if consent is missing so you don't burn the gateway provisioning time.

If this cell raises, forward the printed `az ad sp create` command to a tenant admin, wait for them to run it, then rerun this cell.


In [ ]:
import subprocess
import sys
from pathlib import Path

import requests
from azure.identity import DefaultAzureCredential

demo_root = Path("..").resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

from app.notebook_helpers import resolve_az_cli

# `az account show` is kept as a small bootstrap call to read the active
# subscription + tenant ID. There is no clean SDK path to "the subscription the
# user picked in az login" without re-implementing CLI context resolution, and
# the repo convention forbids assuming the first enabled subscription.
az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found.")
import json as _json
account = _json.loads(subprocess.run([az_cmd, "account", "show", "-o", "json"], capture_output=True, text=True, check=True).stdout)
subscription_id = account["id"]
tenant_id = account["tenantId"]

VPN_CLIENT_APP_ID = "c632b3df-fb67-4d84-bdcf-b95ad541b5c8"

# Check VPN client SP existence via Microsoft Graph REST (replaces `az ad sp show`).
credential = DefaultAzureCredential()
graph_token = credential.get_token("https://graph.microsoft.com/.default").token
sp_resp = requests.get(
    f"https://graph.microsoft.com/v1.0/servicePrincipals?$filter=appId eq '{VPN_CLIENT_APP_ID}'",
    headers={"Authorization": f"Bearer {graph_token}"},
    timeout=30,
)
sp_resp.raise_for_status()
if not sp_resp.json().get("value"):
    raise RuntimeError(
        "Azure VPN Client app is NOT consented in this tenant. The VPN gateway would deploy successfully but no user could connect.\n"
        f"Forward this command to a tenant admin:\n"
        f"  az ad sp create --id {VPN_CLIENT_APP_ID} --tenant {tenant_id}\n"
        f"Then rerun this cell."
    )
print(f"✓ Azure VPN Client consented in tenant {tenant_id}")
print(f"Subscription: {subscription_id}")


## Step 1 - Set deployment variables


In [ ]:
RESOURCE_GROUP = "rg-search01-private-endpoint-ne"
# Any AZ-capable region with capacity for VpnGw1AZ + Standard AI Search + Standard_B1s works.
# If deployment fails with ResourcesForSkuUnavailable / InsufficientResources, run
#   python ../scripts/check_region_capacity.py
# from the notebooks/ folder and set LOCATION to the first region that comes back green.
LOCATION = "northeurope"
DEMO_ID = "search01-private-endpoints"

print(f"Resource group : {RESOURCE_GROUP}")
print(f"Location       : {LOCATION}")
print(f"Demo ID        : {DEMO_ID}")


## Step 2 - Create resource group


In [ ]:
from azure.mgmt.resource.resources import ResourceManagementClient

resource_client = ResourceManagementClient(credential, subscription_id)
rg = resource_client.resource_groups.create_or_update(RESOURCE_GROUP, {"location": LOCATION})
print(f"Resource group ready: {rg.name} ({rg.location})")


## Step 3 - Compile Bicep, deploy, and capture outputs

`az bicep build` is kept as a local-only compile step (it ships with the CLI; no REST involved). The actual deployment runs via `DeploymentsMgmtClient.deployments.begin_create_or_update` (from `azure-mgmt-resource-deployments`, which split off `azure-mgmt-resource` in v25.0.0), and outputs come back inline from the poller result.

**Redeploy safety:** Azure treats `osProfile.customData` and `linuxConfiguration.ssh.publicKeys` as immutable on single VMs (per [Microsoft Learn](https://learn.microsoft.com/en-us/azure/virtual-machines/custom-data#can-i-update-custom-data-after-the-vm-has-been-created)), so any redeploy that re-PUTs the VM with these fields fails with `PropertyChangeNotAllowed`. This cell checks whether `vm-search01-dns` already exists and passes `deployVm=false` to the Bicep template on redeploy so the VM is skipped entirely — leaving Search / Storage / VPN gateway free to reconcile. The SSH public key is persisted to `env/dns_vm_key.pub` and reused so first-deploy retries stay stable too.


In [ ]:
import json
import tempfile
import time

from azure.core.exceptions import HttpResponseError
from azure.mgmt.resource.deployments import DeploymentsMgmtClient
from azure.mgmt.resource.deployments.models import Deployment, DeploymentMode, DeploymentProperties

# Deployments were split out of azure-mgmt-resource into azure-mgmt-resource-deployments
# in v25.0.0 (2026-02). Use the dedicated DeploymentsMgmtClient.
deployments_client = DeploymentsMgmtClient(credential, subscription_id)

# Signed-in user object ID via Microsoft Graph (replaces `az ad signed-in-user show`).
me_resp = requests.get(
    "https://graph.microsoft.com/v1.0/me",
    headers={"Authorization": f"Bearer {graph_token}"},
    timeout=30,
)
me_resp.raise_for_status()
deployer_principal_id = me_resp.json()["id"]

# Determine whether the DNS forwarder VM already exists in the target RG.
# If it does, we must NOT re-PUT it — Azure treats osProfile.customData and
# linuxConfiguration.ssh.publicKeys as immutable on single VMs, and any
# incremental redeploy that includes the VM resource will fail with
# PropertyChangeNotAllowed. When the VM exists we set deployVm=false so the
# Bicep template skips the VM resource entirely.
VM_NAME = "vm-search01-dns"
try:
    resource_client.resources.get(
        resource_group_name=RESOURCE_GROUP,
        resource_provider_namespace="Microsoft.Compute",
        parent_resource_path="",
        resource_type="virtualMachines",
        resource_name=VM_NAME,
        api_version="2023-09-01",
    )
    vm_exists = True
except HttpResponseError as e:
    if e.status_code != 404:
        raise
    vm_exists = False

deploy_vm = not vm_exists
print(f"VM '{VM_NAME}' exists: {vm_exists} → deployVm={deploy_vm}")

# SSH public key is persisted to env/dns_vm_key.pub so redeploys (and first-deploy
# retries) reuse the same value. The Linux VM requires a public key because we
# set disablePasswordAuthentication: true, but the NSG blocks SSH inbound so the
# key is never used for ingress.
ssh_key_path = Path("../env/dns_vm_key.pub")
ssh_key_path.parent.mkdir(parents=True, exist_ok=True)
if ssh_key_path.exists():
    admin_public_key = ssh_key_path.read_text(encoding="utf-8").strip()
    print(f"Reusing SSH public key from {ssh_key_path.resolve()}")
elif not deploy_vm:
    # VM exists but we don't have the public key locally — Bicep won't consume
    # adminPublicKey when deployVm=false, so an empty string is fine.
    admin_public_key = ""
    print("VM exists and no SSH key file present; passing empty adminPublicKey (unused when deployVm=false).")
else:
    with tempfile.TemporaryDirectory() as tmp:
        key_path = Path(tmp) / "dns_vm_key"
        subprocess.run(["ssh-keygen", "-t", "ed25519", "-N", "", "-f", str(key_path), "-C", "search01-dns", "-q"], check=True)
        admin_public_key = key_path.with_suffix(".pub").read_text(encoding="utf-8").strip()
    ssh_key_path.write_text(admin_public_key + "\n", encoding="utf-8")
    print(f"Generated new SSH public key and wrote to {ssh_key_path.resolve()}")

# Compile main.bicep -> ARM JSON locally. `az bicep build` is a local compile,
# not an API call, and ships with the Azure CLI users already have.
template_path = (Path("..") / "infra" / "main.bicep").resolve()
with tempfile.TemporaryDirectory() as build_dir:
    arm_path = Path(build_dir) / "main.json"
    subprocess.run([az_cmd, "bicep", "build", "--file", str(template_path), "--outfile", str(arm_path)], check=True, capture_output=True, text=True)
    template = json.loads(arm_path.read_text(encoding="utf-8"))

deployment_name = f"search01-{int(time.time())}"
deployment_props = DeploymentProperties(
    mode=DeploymentMode.incremental,
    template=template,
    parameters={
        "location": {"value": LOCATION},
        "demoId": {"value": DEMO_ID},
        "deployerPrincipalId": {"value": deployer_principal_id},
        "tenantId": {"value": tenant_id},
        "adminPublicKey": {"value": admin_public_key},
        "deployVm": {"value": deploy_vm},
    },
)
poller = deployments_client.deployments.begin_create_or_update(
    resource_group_name=RESOURCE_GROUP,
    deployment_name=deployment_name,
    parameters=Deployment(properties=deployment_props),
)
print(f"Deployment {deployment_name} started; waiting (typically 30-45 min for VPN gateway on first deploy)...")
deployment_result = poller.result()
print("Deployment succeeded")

required = ["storageAccountName", "containerName", "storageAccountResourceId", "searchServiceName", "searchEndpoint", "searchResourceId", "vnetGatewayName", "dnsForwarderPrivateIp", "vnetId", "snetPePrefix", "vpnClientAddressPool", "resourceGroupName", "tenantId"]
raw_outputs = deployment_result.properties.outputs or {}
missing = [k for k in required if k not in raw_outputs]
if missing:
    raise RuntimeError(f"Missing outputs: {missing}")
resolved = {k: raw_outputs[k]["value"] for k in required}
for k, v in resolved.items():
    print(f"{k}: {v}")


## Step 4 - Write `env/.env`


In [ ]:
lines = [
    "# Auto-generated by 01_deploy_infra.ipynb - do not commit",
    f"AZURE_SUBSCRIPTION_ID={subscription_id}",
    f"AZURE_TENANT_ID={resolved['tenantId']}",
    f"AZURE_RESOURCE_GROUP={resolved['resourceGroupName']}",
    f"AZURE_LOCATION={LOCATION}",
    f"SEARCH_DEMO_ID={DEMO_ID}",
    "",
    f"STORAGE_ACCOUNT_NAME={resolved['storageAccountName']}",
    f"STORAGE_CONTAINER_NAME={resolved['containerName']}",
    f"STORAGE_ACCOUNT_RESOURCE_ID={resolved['storageAccountResourceId']}",
    "",
    f"SEARCH_SERVICE_NAME={resolved['searchServiceName']}",
    f"SEARCH_ENDPOINT={resolved['searchEndpoint']}",
    f"SEARCH_RESOURCE_ID={resolved['searchResourceId']}",
    "",
    f"VNET_GATEWAY_NAME={resolved['vnetGatewayName']}",
    f"DNS_FORWARDER_PRIVATE_IP={resolved['dnsForwarderPrivateIp']}",
    f"VNET_ID={resolved['vnetId']}",
    f"SNET_PE_PREFIX={resolved['snetPePrefix']}",
    f"VPN_CLIENT_ADDRESS_POOL={resolved['vpnClientAddressPool']}",
]
env_path = Path("../env/.env")
env_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Written {env_path.resolve()}")


## Step 5 - Verify `env/.env`


In [ ]:
env_path = Path("../env/.env")
if not env_path.exists():
    raise FileNotFoundError(env_path)
print(env_path.read_text(encoding="utf-8"))


---

Continue with `02_connect_vpn.ipynb`.
